In [1]:
import json, glob, re, pycm, pandas as pd, numpy as np, scipy.stats as stats
from sklearn.metrics import f1_score
from typing import Tuple, List
from subsampling import subsample_statistic_standard_error
import warnings

In [2]:
RUN_VERSION = "v30"

In [3]:
data = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    df = pd.DataFrame(json.load(open(file, 'r')))
    keys = {
        "Judge Model": model,
        "Prompt": prompt,
        "Dataset": dataset
    }
    n = len(df)
    wk_v_counts = df["wk_v"].value_counts()
    wk_v_freq = (wk_v_counts / n).to_dict()
    I_freq = {
        '<t,t>': 0.0, 
        '<t,e>': 0.0, 
        '<t,f>': 0.0, 
        '<e,t>': 0.0, 
        '<e,e>': 0.0, 
        '<e,f>': 0.0, 
        '<f,t>': 0.0, 
        '<f,e>': 0.0, 
        '<f,f>': 0.0
    }
    I_counts = df["I"].value_counts()
    I_freq_rec = (I_counts / n).to_dict()
    for tv in I_freq_rec:
        I_freq[tv] = I_freq_rec[tv]
    addl_stats = { 
        'Time (mean)': df['execution_time'].mean(),
        'Time (stdev)': df['execution_time'].std(),
        'Tokens (mean)': df['tokens_used'].mean(),
        'Tokens (stdev)': df['tokens_used'].std(),
        'Coverage': (n - wk_v_counts['e']) / n
    }
    data.append({ **keys, **wk_v_freq, **I_freq, **addl_stats })
df1 = pd.DataFrame.from_records(data)
df1 = df1.round(3)
df1 = df1.sort_values(["Judge Model", "Dataset", "Prompt"])
df1

,Judge Model,Prompt,Dataset,e,f,t,"<t,t>","<t,e>","<t,f>","<e,t>","<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
33,claude-3-5-haiku-20241022,baseline,gpqa,0.222,0.248,0.530,0.110,0.000,0.530,0.000,0.0,0.0,0.248,0.000,0.112,30.518,4.094,2641.065,704.475,0.778
14,claude-3-5-haiku-20241022,few,gpqa,0.562,0.225,0.212,0.502,0.000,0.212,0.000,0.0,0.0,0.225,0.000,0.060,47.440,4.693,7673.445,700.470,0.438
19,claude-3-5-haiku-20241022,zero,gpqa,0.588,0.205,0.208,0.530,0.000,0.208,0.000,0.0,0.0,0.205,0.000,0.058,43.120,3.595,4221.732,685.849,0.412
34,claude-3-5-haiku-20241022,baseline,simpleqa,0.542,0.175,0.282,0.035,0.000,0.282,0.000,0.0,0.0,0.175,0.000,0.507,15.814,3.074,1014.975,149.236,0.458
26,claude-3-5-haiku-20241022,few,simpleqa,0.550,0.288,0.162,0.132,0.000,0.162,0.000,0.0,0.0,0.288,0.000,0.418,40.007,4.093,6435.110,158.399,0.450
31,claude-3-5-haiku-20241022,zero,simpleqa,0.615,0.240,0.145,0.148,0.000,0.145,0.000,0.0,0.0,0.240,0.000,0.468,38.918,3.143,3095.572,131.718,0.385
24,claude-3-5-sonnet-20241022,baseline,gpqa,0.252,0.355,0.392,0.202,0.000,0.392,0.000,0.0,0.0,0.355,0.000,0.050,34.223,6.079,2914.805,778.985,0.748
4,claude-3-5-sonnet-20241022,few,gpqa,0.460,0.360,0.180,0.448,0.000,0.180,0.000,0.0,0.0,0.360,0.000,0.012,52.923,6.410,8079.405,745.709,0.540
17,claude-3-5-sonnet-20241022,zero,gpqa,0.458,0.325,0.218,0.435,0.000,0.218,0.000,0.0,0.0,0.325,0.000,0.022,53.912,6.412,4863.482,731.269,0.542
18,claude-3-5-sonnet-20241022,baseline,simpleqa,0.498,0.285,0.218,0.052,0.000,0.218,0.000,0.0,0.0,0.285,0.000,0.445,19.217,4.493,1190.432,174.003,0.502


In [4]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    cms[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset].F1_Macro, 
        cms[model][prompt][dataset].ACC_Macro, 
        cms[model][prompt][dataset].FPR['t'], 
        cms[model][prompt][dataset].FNR['t'], 
        cms[model][prompt][dataset].F1['t'], 
        cms[model][prompt][dataset].F1['f']
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]
df2 = pd.DataFrame(data, columns=column_names)
df2 = df2.round(3)
df2 = df2.sort_values(["Judge Model", "Dataset", "Prompt"])
df2

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
34,claude-3-5-haiku-20241022,baseline,gpqa,0.578,0.588,0.589,0.216,0.644,0.511
30,claude-3-5-haiku-20241022,few,gpqa,0.604,0.606,0.389,0.400,0.582,0.627
32,claude-3-5-haiku-20241022,zero,gpqa,0.648,0.648,0.367,0.333,0.633,0.663
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.667,0.672,0.444,0.215,0.709,0.625
31,claude-3-5-haiku-20241022,few,simpleqa,0.653,0.661,0.207,0.477,0.601,0.705
33,claude-3-5-haiku-20241022,zero,simpleqa,0.673,0.682,0.217,0.437,0.620,0.726
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.712,0.712,0.313,0.262,0.719,0.705
18,claude-3-5-sonnet-20241022,few,gpqa,0.716,0.727,0.130,0.436,0.659,0.772
20,claude-3-5-sonnet-20241022,zero,gpqa,0.738,0.742,0.170,0.352,0.708,0.769
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.810,0.811,0.127,0.253,0.796,0.824


In [5]:
df = df2.merge(df1, on=['Judge Model', 'Prompt', 'Dataset'], how='inner')
df

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-),e,...,"<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
0,claude-3-5-haiku-20241022,baseline,gpqa,0.578,0.588,0.589,0.216,0.644,0.511,0.222,...,0.0,0.0,0.248,0.000,0.112,30.518,4.094,2641.065,704.475,0.778
1,claude-3-5-haiku-20241022,few,gpqa,0.604,0.606,0.389,0.400,0.582,0.627,0.562,...,0.0,0.0,0.225,0.000,0.060,47.440,4.693,7673.445,700.470,0.438
2,claude-3-5-haiku-20241022,zero,gpqa,0.648,0.648,0.367,0.333,0.633,0.663,0.588,...,0.0,0.0,0.205,0.000,0.058,43.120,3.595,4221.732,685.849,0.412
3,claude-3-5-haiku-20241022,baseline,simpleqa,0.667,0.672,0.444,0.215,0.709,0.625,0.542,...,0.0,0.0,0.175,0.000,0.507,15.814,3.074,1014.975,149.236,0.458
4,claude-3-5-haiku-20241022,few,simpleqa,0.653,0.661,0.207,0.477,0.601,0.705,0.550,...,0.0,0.0,0.288,0.000,0.418,40.007,4.093,6435.110,158.399,0.450
5,claude-3-5-haiku-20241022,zero,simpleqa,0.673,0.682,0.217,0.437,0.620,0.726,0.615,...,0.0,0.0,0.240,0.000,0.468,38.918,3.143,3095.572,131.718,0.385
6,claude-3-5-sonnet-20241022,baseline,gpqa,0.712,0.712,0.313,0.262,0.719,0.705,0.252,...,0.0,0.0,0.355,0.000,0.050,34.223,6.079,2914.805,778.985,0.748
7,claude-3-5-sonnet-20241022,few,gpqa,0.716,0.727,0.130,0.436,0.659,0.772,0.460,...,0.0,0.0,0.360,0.000,0.012,52.923,6.410,8079.405,745.709,0.540
8,claude-3-5-sonnet-20241022,zero,gpqa,0.738,0.742,0.170,0.352,0.708,0.769,0.458,...,0.0,0.0,0.325,0.000,0.022,53.912,6.412,4863.482,731.269,0.542
9,claude-3-5-sonnet-20241022,baseline,simpleqa,0.810,0.811,0.127,0.253,0.796,0.824,0.498,...,0.0,0.0,0.285,0.000,0.445,19.217,4.493,1190.432,174.003,0.502


In [6]:
df_evaluation = df[["Dataset", "Judge Model", "Prompt", "Coverage", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]].copy()
df_evaluation = df_evaluation.sort_values(["Dataset", "Judge Model", "Prompt"])
df_evaluation

,Dataset,Judge Model,Prompt,Coverage,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
0,gpqa,claude-3-5-haiku-20241022,baseline,0.778,0.578,0.588,0.589,0.216,0.644,0.511
1,gpqa,claude-3-5-haiku-20241022,few,0.438,0.604,0.606,0.389,0.400,0.582,0.627
2,gpqa,claude-3-5-haiku-20241022,zero,0.412,0.648,0.648,0.367,0.333,0.633,0.663
6,gpqa,claude-3-5-sonnet-20241022,baseline,0.748,0.712,0.712,0.313,0.262,0.719,0.705
7,gpqa,claude-3-5-sonnet-20241022,few,0.540,0.716,0.727,0.130,0.436,0.659,0.772
8,gpqa,claude-3-5-sonnet-20241022,zero,0.542,0.738,0.742,0.170,0.352,0.708,0.769
12,gpqa,llama-4-maverick,baseline,0.852,0.774,0.774,0.260,0.188,0.772,0.777
13,gpqa,llama-4-maverick,few,0.805,0.751,0.752,0.294,0.201,0.760,0.742
14,gpqa,llama-4-maverick,zero,0.618,0.765,0.773,0.175,0.298,0.723,0.808
18,gpqa,llama-4-scout,baseline,0.712,0.702,0.702,0.329,0.262,0.693,0.710


In [7]:
df_evaluation.mean(numeric_only=True)

Coverage    0.575806
Macro-F1    0.658750
Acc.        0.682917
FPR         0.287500
FNR         0.372111
F1 (+)      0.626944
F1 (-)      0.690500
dtype: float64

In [8]:
df_evaluation[df_evaluation["Dataset"] == "gpqa"].mean(numeric_only=True)

Coverage    0.603667
Macro-F1    0.646833
Acc.        0.674333
FPR         0.259389
FNR         0.417889
F1 (+)      0.586778
F1 (-)      0.706833
dtype: float64

In [9]:
df_evaluation[df_evaluation["Dataset"] == "simpleqa"].mean(numeric_only=True)

Coverage    0.547944
Macro-F1    0.670667
Acc.        0.691500
FPR         0.315611
FNR         0.326333
F1 (+)      0.667111
F1 (-)      0.674167
dtype: float64

In [10]:
evaluation_table_latex = df_evaluation.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Evaluation of different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:expeval",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(evaluation_table_latex, file=open("paper/evaluation_table.tex", "w+"))


In [11]:
df_truth_value_distribution = df[["Dataset", "Judge Model", "Prompt", "Macro-F1", '<t,t>', '<t,f>', '<f,t>', '<f,f>']].copy()
df_truth_value_distribution = df_truth_value_distribution.sort_values(["Dataset", "Judge Model", "Prompt"])
df_truth_value_distribution

,Dataset,Judge Model,Prompt,Macro-F1,"<t,t>","<t,f>","<f,t>","<f,f>"
0,gpqa,claude-3-5-haiku-20241022,baseline,0.578,0.110,0.530,0.248,0.112
1,gpqa,claude-3-5-haiku-20241022,few,0.604,0.502,0.212,0.225,0.060
2,gpqa,claude-3-5-haiku-20241022,zero,0.648,0.530,0.208,0.205,0.058
6,gpqa,claude-3-5-sonnet-20241022,baseline,0.712,0.202,0.392,0.355,0.050
7,gpqa,claude-3-5-sonnet-20241022,few,0.716,0.448,0.180,0.360,0.012
8,gpqa,claude-3-5-sonnet-20241022,zero,0.738,0.435,0.218,0.325,0.022
12,gpqa,llama-4-maverick,baseline,0.774,0.040,0.442,0.410,0.108
13,gpqa,llama-4-maverick,few,0.751,0.155,0.438,0.368,0.040
14,gpqa,llama-4-maverick,zero,0.765,0.370,0.245,0.372,0.012
18,gpqa,llama-4-scout,baseline,0.702,0.072,0.368,0.345,0.215


In [12]:
df_truth_value_distribution.mean(numeric_only=True)

Macro-F1    0.658750
<t,t>       0.305556
<t,f>       0.269028
<f,t>       0.306806
<f,f>       0.118444
dtype: float64

In [13]:
df_truth_value_distribution.std(numeric_only=True)

Macro-F1    0.095937
<t,t>       0.195994
<t,f>       0.151290
<f,t>       0.112200
<f,f>       0.142787
dtype: float64

In [14]:
df_truth_value_distribution[df_truth_value_distribution["Dataset"] == "gpqa"].mean(numeric_only=True)

Macro-F1    0.646833
<t,t>       0.333778
<t,f>       0.259722
<f,t>       0.344056
<f,f>       0.062500
dtype: float64

In [15]:
df_truth_value_distribution[df_truth_value_distribution["Dataset"] == "simpleqa"].mean(numeric_only=True)

Macro-F1    0.670667
<t,t>       0.277333
<t,f>       0.278333
<f,t>       0.269556
<f,f>       0.174389
dtype: float64

In [16]:
truth_value_distribution_table_latex = df_truth_value_distribution.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Distribution of bilateral truth values different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:exptvdist",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(truth_value_distribution_table_latex, file=open("paper/truth_value_distribution_table.tex", "w+"))

In [17]:
df_cost = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()
df_cost["Mean Time"] = df_cost["Time (mean)"].combine(df_cost["Time (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost["Mean Tokens Used"] = df_cost["Tokens (mean)"].combine(df_cost["Tokens (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost = df_cost[["Dataset", "Judge Model", "Prompt", 'Mean Time', 'Mean Tokens Used']]
df_cost = df_cost.sort_values(["Dataset", "Judge Model", "Prompt"])
df_cost

,Dataset,Judge Model,Prompt,Mean Time,Mean Tokens Used
0,gpqa,claude-3-5-haiku-20241022,baseline,30.52 (4.09),2641.07 (704.48)
1,gpqa,claude-3-5-haiku-20241022,few,47.44 (4.69),7673.44 (700.47)
2,gpqa,claude-3-5-haiku-20241022,zero,43.12 (3.60),4221.73 (685.85)
6,gpqa,claude-3-5-sonnet-20241022,baseline,34.22 (6.08),2914.80 (778.99)
7,gpqa,claude-3-5-sonnet-20241022,few,52.92 (6.41),8079.40 (745.71)
8,gpqa,claude-3-5-sonnet-20241022,zero,53.91 (6.41),4863.48 (731.27)
12,gpqa,llama-4-maverick,baseline,65.91 (52.53),6225.19 (1526.74)
13,gpqa,llama-4-maverick,few,65.08 (41.43),9945.70 (1444.02)
14,gpqa,llama-4-maverick,zero,75.95 (44.90),7492.54 (1357.27)
18,gpqa,llama-4-scout,baseline,63.24 (28.92),5403.00 (1833.85)


In [18]:
df_cost_numeric = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()

In [19]:
df_cost_numeric.mean(numeric_only=True)

Time (mean)         38.289833
Time (stdev)        14.892167
Tokens (mean)     4915.716444
Tokens (stdev)     723.483611
dtype: float64

In [20]:
df_cost_numeric[(df_cost_numeric["Judge Model"] != "llama-4-scout") & (df_cost_numeric["Judge Model"] != "llama-4-maverick")].mean(numeric_only=True)

Time (mean)         31.205250
Time (stdev)         7.144208
Tokens (mean)     4437.153583
Tokens (stdev)     588.382042
dtype: float64

In [21]:
(300. * 30. * 6. * 3. * 2.) / (24. * 60. * 60.) # days to process

3.75

In [22]:
(300 * 4354 * 6 * 3 * 2) # total tokens

47023200

In [23]:
cost_table_latex = df_cost.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:6.6g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Execution time and tokens used by different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:expcosts",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(cost_table_latex, file=open("paper/cost_table.tex", "w+"))

In [24]:
cms_wk = {}
cms_upper = {}
cms_lower = {}
choices = ["t", "f"]
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms_wk:
        cms_wk[model] = {}
    if prompt not in cms_wk[model]:
        cms_wk[model][prompt] = {}
    if model not in cms_upper:
        cms_upper[model] = {}
    if prompt not in cms_upper[model]:
        cms_upper[model][prompt] = {}
    if model not in cms_lower:
        cms_lower[model] = {}
    if prompt not in cms_lower[model]:
        cms_lower[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    upper_conditions = [ (df["I_0"] == "t") & (df["I_1"] != "e"), (df["I_0"] == "f") & (df["I_1"] != "e") ]
    lower_conditions = [ (df["I_1"] == "f") & (df["I_0"] != "e"), (df["I_1"] == "t") & (df["I_0"] != "e") ]
    df["wk_v_upper"] = np.select(upper_conditions, choices, default="e")
    df["wk_v_lower"] = np.select(lower_conditions, choices, default="e")
    cms_wk[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])
    cms_upper[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v_upper"].tolist(), digit=2, classes=[ 't', 'f' ])
    cms_lower[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v_lower"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms_wk[model][prompt][dataset].F1_Macro, 
        cms_wk[model][prompt][dataset].POP['t'] / len(df), 
        cms_upper[model][prompt][dataset].F1_Macro, 
        cms_upper[model][prompt][dataset].POP['t'] / len(df), 
        cms_lower[model][prompt][dataset].F1_Macro, 
        cms_lower[model][prompt][dataset].POP['t'] / len(df)
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Coverage", "Paraconsistent Macro-F1", "Paraconsistent Coverage", "Paracomplete Macro-F1", "Paracomplete Coverage"]
df_approx = pd.DataFrame(data, columns=column_names)
df_approx = df_approx.round(3)
df_approx = df_approx[["Dataset", "Judge Model", "Prompt", "Macro-F1", "Coverage", "Paraconsistent Macro-F1", "Paracomplete Macro-F1"]].copy()
df_approx = df_approx.sort_values(["Dataset", "Judge Model", "Prompt"])
df_approx

,Dataset,Judge Model,Prompt,Macro-F1,Coverage,Paraconsistent Macro-F1,Paracomplete Macro-F1
34,gpqa,claude-3-5-haiku-20241022,baseline,0.578,0.778,0.556,0.568
30,gpqa,claude-3-5-haiku-20241022,few,0.604,0.438,0.524,0.521
32,gpqa,claude-3-5-haiku-20241022,zero,0.648,0.412,0.538,0.533
23,gpqa,claude-3-5-sonnet-20241022,baseline,0.712,0.748,0.620,0.693
18,gpqa,claude-3-5-sonnet-20241022,few,0.716,0.540,0.625,0.570
20,gpqa,claude-3-5-sonnet-20241022,zero,0.738,0.542,0.608,0.618
9,gpqa,llama-4-maverick,baseline,0.774,0.852,0.745,0.722
10,gpqa,llama-4-maverick,few,0.751,0.805,0.698,0.705
6,gpqa,llama-4-maverick,zero,0.765,0.618,0.690,0.619
24,gpqa,llama-4-scout,baseline,0.702,0.712,0.630,0.654


In [25]:
approx_table_latex = df_approx.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Comparison of Macro-F1 scores of differing epistemic policies.",        # Add a caption to your table
    label="tab:expapprox",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(approx_table_latex, file=open("paper/approx_table.tex", "w+"))


In [26]:
df_approx.mean(numeric_only=True)

Macro-F1                   0.658750
Coverage                   0.575806
Paraconsistent Macro-F1    0.596333
Paracomplete Macro-F1      0.572278
dtype: float64

In [27]:
df_approx.std(numeric_only=True)

Macro-F1                   0.095937
Coverage                   0.138027
Paraconsistent Macro-F1    0.056214
Paracomplete Macro-F1      0.082816
dtype: float64

In [29]:
bi_f1 = df_approx['Macro-F1']
uni_f1 = df_approx['Paraconsistent Macro-F1']
stat, p = stats.mannwhitneyu(uni_f1, bi_f1, alternative='less', method='auto')
print(f'U-statistic: {stat}, p-value: {p:.5f}')

U-statistic: 330.0, p-value: 0.00017


In [30]:
bi_f1 = df_approx['Macro-F1']
uni_f1 = df_approx['Paracomplete Macro-F1']
stat, p = stats.mannwhitneyu(uni_f1, bi_f1, alternative='less', method='auto')
print(f'U-statistic: {stat}, p-value: {p:.5f}')

U-statistic: 287.0, p-value: 0.00002


In [31]:
upper_f1 = df_approx['Paraconsistent Macro-F1']
lower_f1 = df_approx['Paracomplete Macro-F1']
stat, p = stats.mannwhitneyu(lower_f1, upper_f1, alternative='less', method='auto')
print(f'U-statistic: {stat}, p-value: {p:.5f}')

U-statistic: 537.5, p-value: 0.10768


In [32]:
df_approx[df_approx["Dataset"] == "gpqa"].mean(numeric_only=True)

Macro-F1                   0.646833
Coverage                   0.603667
Paraconsistent Macro-F1    0.601278
Paracomplete Macro-F1      0.570167
dtype: float64

In [33]:
df_approx[df_approx["Dataset"] == "simpleqa"].mean(numeric_only=True)

Macro-F1                   0.670667
Coverage                   0.547944
Paraconsistent Macro-F1    0.591389
Paracomplete Macro-F1      0.574389
dtype: float64

In [34]:
warnings.filterwarnings('ignore')

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute macro F1 score.
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        Macro F1 score
    """
    return f1_score(y_true, y_pred, average='macro')

def subsample_standard_error(y_true: np.ndarray, 
                           y_pred: np.ndarray,
                           subsample_size: int = None,
                           n_subsamples: int = 1000,
                           random_state: int = None) -> Tuple[float, float, List[float]]:
    """
    Estimate standard error of macro F1 using subsampling method from Politis & Romano (1994).
    
    Args:
        y_true: Ground truth binary labels (n,)
        y_pred: Predicted binary labels (n,)
        subsample_size: Size of each subsample (b). If None, uses b = n^(2/3)
        n_subsamples: Number of subsamples to draw
        random_state: Random seed for reproducibility
        
    Returns:
        Tuple of (original_f1, estimated_std_error, subsample_f1_scores)
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    n = len(y_true)
    
    # Choose subsample size according to theory: b → ∞ and b/n → 0
    # Common choice is b = n^(2/3) for optimal rate
    if subsample_size is None:
        subsample_size = int(np.power(n, 2/3))
    
    # Ensure subsample size is valid
    subsample_size = min(subsample_size, n-1)
    subsample_size = max(subsample_size, 2)  # Need at least 2 samples for F1
    
    # print(f"Sample size n = {n}")
    # print(f"Subsample size b = {subsample_size}")
    # print(f"Ratio b/n = {subsample_size/n:.3f}")
    # print(f"Number of subsamples = {n_subsamples}")
    
    # Compute original statistic
    original_f1 = macro_f1_score(y_true, y_pred)
    
    # Generate subsamples and compute F1 scores
    subsample_f1_scores = []
    
    for i in range(n_subsamples):
        # Sample without replacement
        indices = np.random.choice(n, size=subsample_size, replace=False)
        y_true_sub = y_true[indices]
        y_pred_sub = y_pred[indices]
        
        # Compute F1 on subsample
        try:
            f1_sub = macro_f1_score(y_true_sub, y_pred_sub)
            subsample_f1_scores.append(f1_sub)
        except:
            # Skip if F1 cannot be computed (e.g., if one class is missing)
            continue
    
    subsample_f1_scores = np.array(subsample_f1_scores)
    
    # According to the paper, we need to properly normalize
    # The standard error is estimated from the variance of the subsample statistics
    # For the empirical variance, we use the sample variance of subsample F1 scores
    
    # Estimate standard error
    # This approximates the standard error of the original F1 score
    estimated_std_error = np.sqrt(subsample_size / n) * np.std(subsample_f1_scores, ddof=1)
    
    # print(f"\nResults:")
    # print(f"Original macro F1: {original_f1:.4f}")
    # print(f"Mean subsample F1: {np.mean(subsample_f1_scores):.4f}")
    # print(f"Std of subsample F1: {np.std(subsample_f1_scores, ddof=1):.4f}")
    # print(f"Estimated standard error: {estimated_std_error:.4f}")
    # print(f"Number of valid subsamples: {len(subsample_f1_scores)}")
    
    return original_f1, estimated_std_error, subsample_f1_scores.tolist()

def confidence_interval(f1_score: float, 
                       std_error: float, 
                       confidence_level: float = 0.95) -> Tuple[float, float]:
    """
    Construct confidence interval for F1 score using normal approximation.
    
    Args:
        f1_score: Original F1 score
        std_error: Estimated standard error
        confidence_level: Confidence level (default 0.95 for 95% CI)
        
    Returns:
        Tuple of (lower_bound, upper_bound)
    """
    from scipy.stats import norm
    
    alpha = 1 - confidence_level
    z_score = norm.ppf(1 - alpha/2)
    
    margin_of_error = z_score * std_error
    lower_bound = max(0, f1_score - margin_of_error)  # F1 is bounded by 0
    upper_bound = min(1, f1_score + margin_of_error)  # F1 is bounded by 1
    
    return lower_bound, upper_bound

def analyze_classification_results(df, 
                                 true_label_col='ground_truth', 
                                 pred_label_col='predictions',
                                 subsample_size=None,
                                 n_subsamples=1000,
                                 random_state=42):
    """
    Apply subsampling standard error estimation to your classification DataFrame.
    
    Args:
        df: pandas DataFrame with your classification results
        true_label_col: name of column containing ground truth labels
        pred_label_col: name of column containing predictions
        subsample_size: size of subsamples (if None, uses n^(2/3))
        n_subsamples: number of subsamples to draw
        random_state: random seed
        
    Returns:
        Dictionary with results
    """
    
    # Extract arrays from DataFrame
    y_true = df[true_label_col].values
    y_pred = df[pred_label_col].values
    
    # Ensure binary values
    assert set(np.unique(y_true)).issubset({0, 1}), "Ground truth must be binary (0/1)"
    assert set(np.unique(y_pred)).issubset({0, 1}), "Predictions must be binary (0/1)"
    
    # print(f"Analyzing {len(df)} classification results...")
    # print(f"Class distribution in ground truth: {np.bincount(y_true)}")
    # print(f"Class distribution in predictions: {np.bincount(y_pred)}")
    
    # Apply subsampling method
    f1_score, std_error, subsample_scores = subsample_standard_error(
        y_true, y_pred,
        subsample_size=subsample_size,
        n_subsamples=n_subsamples,
        random_state=random_state
    )
    
    # Get confidence interval
    ci_lower, ci_upper = confidence_interval(f1_score, std_error, 0.95)
    
    results = {
        'macro_f1': f1_score,
        'standard_error': std_error,
        'confidence_interval_95': (ci_lower, ci_upper),
        'subsample_scores': subsample_scores,
        'n_valid_subsamples': len(subsample_scores)
    }
    
    return results

# Example usage with your DataFrame:
def example_usage():
    """
    Example of how to use with your DataFrame
    """
    # If your DataFrame looks like this:
    #   ground_truth  predictions
    # 0            1            1
    # 1            0            1
    # 2            1            0
    # ... etc for 400 rows

    # Load your data (replace with your actual loading method)
    # df = pd.read_csv('your_data.csv')
    
    # Create sample data for demonstration
    np.random.seed(42)
    n = 400
    sample_data = {
        'ground_truth': np.random.choice([0, 1], size=n, p=[0.3, 0.7]),
        'predictions': np.random.choice([0, 1], size=n, p=[0.4, 0.6])
    }
    df = pd.DataFrame(sample_data)
    
    # Run the analysis
    results = analyze_classification_results(
        df, 
        true_label_col='ground_truth',    # adjust column name as needed
        pred_label_col='predictions',     # adjust column name as needed
        n_subsamples=1000,               # as you requested
        random_state=42                  # for reproducibility
    )
    
    # Print results
    print(f"\n=== FINAL RESULTS ===")
    print(f"Macro F1 Score: {results['macro_f1']:.4f}")
    print(f"Standard Error: {results['standard_error']:.4f}")
    print(f"95% CI: [{results['confidence_interval_95'][0]:.4f}, {results['confidence_interval_95'][1]:.4f}]")
    
    return results

# Alternative: if you want to experiment with different subsample sizes
def compare_subsample_sizes(df, true_label_col, pred_label_col, sizes=None):
    """
    Compare results using different subsample sizes to see sensitivity.
    """
    if sizes is None:
        n = len(df)
        sizes = [
            int(n**0.5),     # n^(1/2) 
            int(n**(2/3)),   # n^(2/3) - theoretical optimum
            int(n**0.8),     # n^(4/5)
            n//4,            # n/4
            n//3             # n/3
        ]
    
    results = {}
    y_true = df[true_label_col].values
    y_pred = df[pred_label_col].values
    
    for size in sizes:
        if size >= len(df) or size < 10:
            continue
            
        print(f"\n--- Subsample size: {size} (ratio: {size/len(df):.3f}) ---")
        f1, se, _ = subsample_standard_error(
            y_true, y_pred, 
            subsample_size=size, 
            n_subsamples=1000,
            random_state=42
        )
        results[size] = {'f1': f1, 'std_error': se}
    
    return results

if __name__ == "__main__":
    # Run the example
    example_results = example_usage()


=== FINAL RESULTS ===
Macro F1 Score: 0.4483
Standard Error: 0.0229
95% CI: [0.4034, 0.4932]


In [118]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df = df[(df["wk_v"] == 't') | (df["wk_v"] == 'f')].copy()
    df["ground_truth"] = (df["label"] == 't').astype(int)
    df["predictions"] = (df["wk_v"] == 't').astype(int)
    cms[model][prompt][dataset] = analyze_classification_results(df, true_label_col='ground_truth', pred_label_col='predictions', n_subsamples=1000, random_state=9331)

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset]['macro_f1'], 
        cms[model][prompt][dataset]['standard_error'],
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "SE"]
df_subsamples = pd.DataFrame(data, columns=column_names)
df_subsamples = df_subsamples.round(3)
df_subsamples = df_subsamples.sort_values(["Judge Model", "Prompt", "Dataset"]).copy()
# df_subsamples = df_subsamples[["Judge Model", "Prompt", "Dataset", "Macro-F1", "SE"]]
df_subsamples

,Judge Model,Prompt,Dataset,Macro-F1,SE
34,claude-3-5-haiku-20241022,baseline,gpqa,0.578,0.027
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.667,0.033
30,claude-3-5-haiku-20241022,few,gpqa,0.604,0.034
31,claude-3-5-haiku-20241022,few,simpleqa,0.653,0.033
32,claude-3-5-haiku-20241022,zero,gpqa,0.648,0.034
33,claude-3-5-haiku-20241022,zero,simpleqa,0.673,0.036
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.712,0.023
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.810,0.025
18,claude-3-5-sonnet-20241022,few,gpqa,0.716,0.028
19,claude-3-5-sonnet-20241022,few,simpleqa,0.654,0.031


In [119]:
# Assuming your dataframe is called 'df'
# If reading from a file, uncomment and modify the appropriate line below:
# df = pd.read_csv('your_file.csv')
# df = pd.read_excel('your_file.xlsx')

def transform_dataframe(df):
    """
    Transform the dataframe from long format to wide format.
    
    Original format: Judge Model, Prompt, Dataset, Macro-F1, SE
    New format: Judge Model, Prompt, GQPA_F1, GQPA_SE, SimpleQA_F1, SimpleQA_SE
    """
    
    # Create the pivot table
    # We'll pivot on the Dataset column to create separate columns for each dataset
    transformed_df = df.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Macro-F1', 'SE'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()
    
    # Flatten the multi-level column headers
    # This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
    transformed_df.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                             for col in transformed_df.columns]
    
    # Rename columns to match your desired format
    column_mapping = {
        'Judge Model': 'Judge Model',
        'Prompt': 'Prompt',
        'gpqa_Macro-F1': 'GPQA_F1',
        'gpqa_SE': 'GPQA_F1_SE',
        'simpleqa_Macro-F1': 'SimpleQA_F1',
        'simpleqa_SE': 'SimpleQA_F1_SE'
    }
    
    transformed_df = transformed_df.rename(columns=column_mapping)
    
    # Reorder columns to match your specification
    final_columns = ['Judge Model', 'Prompt', 'GPQA_F1', 'GPQA_F1_SE', 'SimpleQA_F1', 'SimpleQA_F1_SE']
    transformed_df = transformed_df[final_columns]
    
    return transformed_df

# Apply the transformation
# transformed_df = transform_dataframe(df)

# Display the result
# print(transformed_df)

# Save to file if needed
# transformed_df.to_csv('transformed_data.csv', index=False)
# transformed_df.to_excel('transformed_data.xlsx', index=False)

# Example of what the output will look like:
"""
Expected output format:
        Judge Model  Prompt  GQPA_F1  GQPA_SE  SimpleQA_F1  SimpleQA_SE
0   claude-3-5-haiku-20241022  baseline    0.578    0.027        0.667      0.033
1   claude-3-5-haiku-20241022       few    0.604    0.034        0.653      0.033
2   claude-3-5-haiku-20241022      zero    0.648    0.034        0.673      0.036
3  claude-3-5-sonnet-20241022  baseline    0.712    0.023        0.810      0.025
...
"""

'\nExpected output format:\n        Judge Model  Prompt  GQPA_F1  GQPA_SE  SimpleQA_F1  SimpleQA_SE\n0   claude-3-5-haiku-20241022  baseline    0.578    0.027        0.667      0.033\n1   claude-3-5-haiku-20241022       few    0.604    0.034        0.653      0.033\n2   claude-3-5-haiku-20241022      zero    0.648    0.034        0.673      0.036\n3  claude-3-5-sonnet-20241022  baseline    0.712    0.023        0.810      0.025\n...\n'

In [120]:
df_transformed_f1 = transform_dataframe(df_subsamples)

In [121]:
tv_stats = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in tv_stats:
        tv_stats[model] = {}
    if prompt not in tv_stats[model]:
        tv_stats[model][prompt] = {}
    if dataset not in tv_stats[model][prompt]:
        tv_stats[model][prompt][dataset] = {}
    tvs = [ "<t,t>", "<t,f>", "<t,e>", "<f,t>", "<f,f>", "<f,e>", "<e,t>", "<e,f>", "<e,e>" ]
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    for tv in tvs:
        df[tv] = (df["I"] == tv).astype(int)
    for tv in tvs:
        tv_stats[model][prompt][dataset][tv] = subsample_statistic_standard_error(df[tv].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)

data = [ 
    [
        model, 
        prompt, 
        dataset, 
        tv,
        tv_stats[model][prompt][dataset][tv][0], 
        tv_stats[model][prompt][dataset][tv][1]
    ] 
    for model in tv_stats 
    for prompt in tv_stats[model] 
    for dataset in tv_stats[model][prompt]
    for tv in tv_stats[model][prompt][dataset]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Truth Value", "Pr.", "SE"]
df_tv_dist_2 = pd.DataFrame(data, columns=column_names)
df_tv_dist_2 = df_tv_dist_2.round(3)
df_tv_dist_2 = df_tv_dist_2.sort_values(["Judge Model", "Prompt", "Dataset"]).copy()
df_tv_dist_2

,Judge Model,Prompt,Dataset,Truth Value,Pr.,SE
306,claude-3-5-haiku-20241022,baseline,gpqa,"<t,t>",0.110,0.015
307,claude-3-5-haiku-20241022,baseline,gpqa,"<t,f>",0.530,0.023
308,claude-3-5-haiku-20241022,baseline,gpqa,"<t,e>",0.000,0.000
309,claude-3-5-haiku-20241022,baseline,gpqa,"<f,t>",0.248,0.020
310,claude-3-5-haiku-20241022,baseline,gpqa,"<f,f>",0.112,0.015
...,...,...,...,...,...,...
4,nf-gpt-4o-mini,zero,simpleqa,"<f,f>",0.002,0.002
5,nf-gpt-4o-mini,zero,simpleqa,"<f,e>",0.000,0.000
6,nf-gpt-4o-mini,zero,simpleqa,"<e,t>",0.000,0.000
7,nf-gpt-4o-mini,zero,simpleqa,"<e,f>",0.000,0.000


In [145]:
transformed_df_tv = df_tv_dist_2.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns=['Dataset', 'Truth Value'],                # This will become our column headers
        values=['Pr.', 'SE'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()

# Flatten the multi-level column headers
# This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
transformed_df_tv.columns = [f'{col[1]}_{col[2]}_{col[0]}' if col[1] != '' else col[0] 
                            for col in transformed_df_tv.columns]
    
# Rename columns to match your desired format
# column_mapping = {
#     'Judge Model': 'Judge Model',
#     'Prompt': 'Prompt',
#     'gpqa_<t,t>': 'GPQA_Mean_Time',
#     'gpqa_Mean Tokens Used': 'GPQA_Mean_Tokens',
#     'simpleqa_Mean Time': 'SimpleQA_Mean_Time',
#     'simpleqa_Mean Tokens Used': 'SimpleQA_Mean_Tokens',
# }
    
# transformed_df_tv = transformed_df_tv.rename(columns=column_mapping)
transformed_df_tv


,Judge Model,Prompt,"gpqa_<e,e>_Pr.","gpqa_<e,f>_Pr.","gpqa_<e,t>_Pr.","gpqa_<f,e>_Pr.","gpqa_<f,f>_Pr.","gpqa_<f,t>_Pr.","gpqa_<t,e>_Pr.","gpqa_<t,f>_Pr.",...,"gpqa_<t,t>_SE","simpleqa_<e,e>_SE","simpleqa_<e,f>_SE","simpleqa_<e,t>_SE","simpleqa_<f,e>_SE","simpleqa_<f,f>_SE","simpleqa_<f,t>_SE","simpleqa_<t,e>_SE","simpleqa_<t,f>_SE","simpleqa_<t,t>_SE"
0,claude-3-5-haiku-20241022,baseline,0.0,0.0,0.0,0.0,0.112,0.248,0.0,0.530,...,0.015,0.0,0.0,0.000,0.000,0.024,0.018,0.000,0.022,0.008
1,claude-3-5-haiku-20241022,few,0.0,0.0,0.0,0.0,0.060,0.225,0.0,0.212,...,0.024,0.0,0.0,0.000,0.000,0.023,0.021,0.000,0.017,0.016
2,claude-3-5-haiku-20241022,zero,0.0,0.0,0.0,0.0,0.058,0.205,0.0,0.208,...,0.023,0.0,0.0,0.000,0.000,0.023,0.020,0.000,0.016,0.016
3,claude-3-5-sonnet-20241022,baseline,0.0,0.0,0.0,0.0,0.050,0.355,0.0,0.392,...,0.018,0.0,0.0,0.000,0.000,0.024,0.021,0.000,0.020,0.010
4,claude-3-5-sonnet-20241022,few,0.0,0.0,0.0,0.0,0.012,0.360,0.0,0.180,...,0.023,0.0,0.0,0.000,0.000,0.012,0.023,0.000,0.014,0.021
5,claude-3-5-sonnet-20241022,zero,0.0,0.0,0.0,0.0,0.022,0.325,0.0,0.218,...,0.023,0.0,0.0,0.000,0.000,0.016,0.022,0.000,0.016,0.020
6,llama-4-maverick,baseline,0.0,0.0,0.0,0.0,0.108,0.410,0.0,0.442,...,0.009,0.0,0.0,0.000,0.000,0.016,0.018,0.000,0.022,0.013
7,llama-4-maverick,few,0.0,0.0,0.0,0.0,0.040,0.368,0.0,0.438,...,0.017,0.0,0.0,0.000,0.000,0.012,0.019,0.000,0.023,0.019
8,llama-4-maverick,zero,0.0,0.0,0.0,0.0,0.012,0.372,0.0,0.245,...,0.022,0.0,0.0,0.002,0.000,0.008,0.018,0.000,0.022,0.024
9,llama-4-scout,baseline,0.0,0.0,0.0,0.0,0.215,0.345,0.0,0.368,...,0.013,0.0,0.0,0.000,0.000,0.021,0.018,0.000,0.022,0.015


In [147]:
for tv in tvs:
    for dataset in ['gpqa', 'simpleqa']:
        transformed_df_tv[f'{dataset} Pr({tv})'] = transformed_df_tv.apply(lambda row: f"{row[f'{dataset}_{tv}_Pr.']} ({row[f'{dataset}_{tv}_SE']})", axis=1)

In [148]:
transformed_df_tv

,Judge Model,Prompt,"gpqa_<e,e>_Pr.","gpqa_<e,f>_Pr.","gpqa_<e,t>_Pr.","gpqa_<f,e>_Pr.","gpqa_<f,f>_Pr.","gpqa_<f,t>_Pr.","gpqa_<t,e>_Pr.","gpqa_<t,f>_Pr.",...,"gpqa Pr(<f,f>)","simpleqa Pr(<f,f>)","gpqa Pr(<f,e>)","simpleqa Pr(<f,e>)","gpqa Pr(<e,t>)","simpleqa Pr(<e,t>)","gpqa Pr(<e,f>)","simpleqa Pr(<e,f>)","gpqa Pr(<e,e>)","simpleqa Pr(<e,e>)"
0,claude-3-5-haiku-20241022,baseline,0.0,0.0,0.0,0.0,0.112,0.248,0.0,0.530,...,0.112 (0.015),0.507 (0.024),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
1,claude-3-5-haiku-20241022,few,0.0,0.0,0.0,0.0,0.060,0.225,0.0,0.212,...,0.06 (0.011),0.418 (0.023),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
2,claude-3-5-haiku-20241022,zero,0.0,0.0,0.0,0.0,0.058,0.205,0.0,0.208,...,0.058 (0.011),0.468 (0.023),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
3,claude-3-5-sonnet-20241022,baseline,0.0,0.0,0.0,0.0,0.050,0.355,0.0,0.392,...,0.05 (0.011),0.445 (0.024),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
4,claude-3-5-sonnet-20241022,few,0.0,0.0,0.0,0.0,0.012,0.360,0.0,0.180,...,0.012 (0.005),0.072 (0.012),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
5,claude-3-5-sonnet-20241022,zero,0.0,0.0,0.0,0.0,0.022,0.325,0.0,0.218,...,0.022 (0.007),0.14 (0.016),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
6,llama-4-maverick,baseline,0.0,0.0,0.0,0.0,0.108,0.410,0.0,0.442,...,0.108 (0.015),0.145 (0.016),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
7,llama-4-maverick,few,0.0,0.0,0.0,0.0,0.040,0.368,0.0,0.438,...,0.04 (0.009),0.068 (0.012),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
8,llama-4-maverick,zero,0.0,0.0,0.0,0.0,0.012,0.372,0.0,0.245,...,0.012 (0.005),0.032 (0.008),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.002 (0.002),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)
9,llama-4-scout,baseline,0.0,0.0,0.0,0.0,0.215,0.345,0.0,0.368,...,0.215 (0.019),0.302 (0.021),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0),0.0 (0.0)


In [150]:
transformed_df_tv = transformed_df_tv[[
    "Judge Model", "Prompt", 
    "gpqa Pr(<t,t>)", "gpqa Pr(<t,f>)", "gpqa Pr(<f,t>)", "gpqa Pr(<f,f>)",
    "simpleqa Pr(<t,t>)", "simpleqa Pr(<t,f>)", "simpleqa Pr(<f,t>)", "simpleqa Pr(<f,f>)"
    ]].copy()

In [151]:
transformed_df_tv

,Judge Model,Prompt,"gpqa Pr(<t,t>)","gpqa Pr(<t,f>)","gpqa Pr(<f,t>)","gpqa Pr(<f,f>)","simpleqa Pr(<t,t>)","simpleqa Pr(<t,f>)","simpleqa Pr(<f,t>)","simpleqa Pr(<f,f>)"
0,claude-3-5-haiku-20241022,baseline,0.11 (0.015),0.53 (0.023),0.248 (0.02),0.112 (0.015),0.035 (0.008),0.282 (0.022),0.175 (0.018),0.507 (0.024)
1,claude-3-5-haiku-20241022,few,0.502 (0.024),0.212 (0.02),0.225 (0.02),0.06 (0.011),0.132 (0.016),0.162 (0.017),0.288 (0.021),0.418 (0.023)
2,claude-3-5-haiku-20241022,zero,0.53 (0.023),0.208 (0.019),0.205 (0.019),0.058 (0.011),0.148 (0.016),0.145 (0.016),0.24 (0.02),0.468 (0.023)
3,claude-3-5-sonnet-20241022,baseline,0.202 (0.018),0.392 (0.022),0.355 (0.022),0.05 (0.011),0.052 (0.01),0.218 (0.02),0.285 (0.021),0.445 (0.024)
4,claude-3-5-sonnet-20241022,few,0.448 (0.023),0.18 (0.018),0.36 (0.022),0.012 (0.005),0.278 (0.021),0.118 (0.014),0.532 (0.023),0.072 (0.012)
5,claude-3-5-sonnet-20241022,zero,0.435 (0.023),0.218 (0.02),0.325 (0.021),0.022 (0.007),0.245 (0.02),0.152 (0.016),0.462 (0.022),0.14 (0.016)
6,llama-4-maverick,baseline,0.04 (0.009),0.442 (0.023),0.41 (0.023),0.108 (0.015),0.088 (0.013),0.588 (0.022),0.18 (0.018),0.145 (0.016)
7,llama-4-maverick,few,0.155 (0.017),0.438 (0.023),0.368 (0.022),0.04 (0.009),0.218 (0.019),0.525 (0.023),0.19 (0.019),0.068 (0.012)
8,llama-4-maverick,zero,0.37 (0.022),0.245 (0.02),0.372 (0.023),0.012 (0.005),0.438 (0.024),0.35 (0.022),0.178 (0.018),0.032 (0.008)
9,llama-4-scout,baseline,0.072 (0.013),0.368 (0.023),0.345 (0.022),0.215 (0.019),0.125 (0.015),0.368 (0.022),0.205 (0.018),0.302 (0.021)


In [152]:
tv_table_latex = transformed_df_tv.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:6.6g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Truth value probabilities for zeta using different judge models and evaluation prompts on two datasets.",        # Add a caption to your table
    label="tab:finaltv",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(tv_table_latex, file=open("paper/tv_table.tex", "w+"))

In [122]:
coverage = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in coverage:
        coverage[model] = {}
    if prompt not in coverage[model]:
        coverage[model][prompt] = {}
    if dataset not in coverage[model][prompt]:
        coverage[model][prompt][dataset] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    df["coverage"] = (df["wk_v"] != 'e').astype(int)
    coverage[model][prompt][dataset] = subsample_statistic_standard_error(df["coverage"].to_numpy(), np.mean, n_subsamples=1000, random_state=9331)

data = [ 
    [
        model, 
        prompt, 
        dataset, 
        coverage[model][prompt][dataset][0], 
        coverage[model][prompt][dataset][1]
    ] 
    for model in coverage 
    for prompt in coverage[model] 
    for dataset in coverage[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Coverage", "SE"]
df_coverage_2 = pd.DataFrame(data, columns=column_names)
df_coverage_2 = df_coverage_2.round(3)
df_coverage_2 = df_coverage_2.sort_values(["Judge Model", "Prompt", "Dataset"]).copy()
df_coverage_2

,Judge Model,Prompt,Dataset,Coverage,SE
34,claude-3-5-haiku-20241022,baseline,gpqa,0.778,0.020
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.458,0.024
30,claude-3-5-haiku-20241022,few,gpqa,0.438,0.024
31,claude-3-5-haiku-20241022,few,simpleqa,0.450,0.023
32,claude-3-5-haiku-20241022,zero,gpqa,0.412,0.023
33,claude-3-5-haiku-20241022,zero,simpleqa,0.385,0.022
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.748,0.020
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.502,0.024
18,claude-3-5-sonnet-20241022,few,gpqa,0.540,0.023
19,claude-3-5-sonnet-20241022,few,simpleqa,0.650,0.022


In [123]:
transformed_df_coverage = df_coverage_2.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Coverage', 'SE'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()

# Flatten the multi-level column headers
# This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
transformed_df_coverage.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                            for col in transformed_df_coverage.columns]
    
# Rename columns to match your desired format
column_mapping = {
    'Judge Model': 'Judge Model',
    'Prompt': 'Prompt',
    'gpqa_Coverage': 'GPQA_Coverage',
    'gpqa_SE': 'GPQA_Coverage_SE',
    'simpleqa_Coverage': 'SimpleQA_Coverage',
    'simpleqa_SE': 'SimpleQA_Coverage_SE'
}
    
transformed_df_coverage = transformed_df_coverage.rename(columns=column_mapping)
transformed_df_coverage


,Judge Model,Prompt,GPQA_Coverage,SimpleQA_Coverage,GPQA_Coverage_SE,SimpleQA_Coverage_SE
0,claude-3-5-haiku-20241022,baseline,0.778,0.458,0.020,0.024
1,claude-3-5-haiku-20241022,few,0.438,0.450,0.024,0.023
2,claude-3-5-haiku-20241022,zero,0.412,0.385,0.023,0.022
3,claude-3-5-sonnet-20241022,baseline,0.748,0.502,0.020,0.024
4,claude-3-5-sonnet-20241022,few,0.540,0.650,0.023,0.022
5,claude-3-5-sonnet-20241022,zero,0.542,0.615,0.024,0.022
6,llama-4-maverick,baseline,0.852,0.768,0.016,0.020
7,llama-4-maverick,few,0.805,0.715,0.018,0.021
8,llama-4-maverick,zero,0.618,0.528,0.022,0.024
9,llama-4-scout,baseline,0.712,0.572,0.021,0.023


In [124]:
df_perf_merged = pd.merge(df_transformed_f1, transformed_df_coverage)

In [125]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1,GPQA_F1_SE,SimpleQA_F1,SimpleQA_F1_SE,GPQA_Coverage,SimpleQA_Coverage,GPQA_Coverage_SE,SimpleQA_Coverage_SE
0,claude-3-5-haiku-20241022,baseline,0.578,0.027,0.667,0.033,0.778,0.458,0.020,0.024
1,claude-3-5-haiku-20241022,few,0.604,0.034,0.653,0.033,0.438,0.450,0.024,0.023
2,claude-3-5-haiku-20241022,zero,0.648,0.034,0.673,0.036,0.412,0.385,0.023,0.022
3,claude-3-5-sonnet-20241022,baseline,0.712,0.023,0.810,0.025,0.748,0.502,0.020,0.024
4,claude-3-5-sonnet-20241022,few,0.716,0.028,0.654,0.031,0.540,0.650,0.023,0.022
5,claude-3-5-sonnet-20241022,zero,0.738,0.029,0.733,0.027,0.542,0.615,0.024,0.022
6,llama-4-maverick,baseline,0.774,0.021,0.673,0.026,0.852,0.768,0.016,0.020
7,llama-4-maverick,few,0.751,0.023,0.692,0.026,0.805,0.715,0.018,0.021
8,llama-4-maverick,zero,0.765,0.025,0.746,0.029,0.618,0.528,0.022,0.024
9,llama-4-scout,baseline,0.702,0.025,0.580,0.031,0.712,0.572,0.021,0.023


In [126]:
df_perf_merged["GPQA_Cov_disp"] = df_perf_merged.apply(lambda row: f'{row["GPQA_Coverage"]} ({row["GPQA_Coverage_SE"]})', axis=1)
df_perf_merged["GPQA_F1_disp"] = df_perf_merged.apply(lambda row: f'{row["GPQA_F1"]} ({row["GPQA_F1_SE"]})', axis=1)
df_perf_merged["SimpleQA_Cov_disp"] = df_perf_merged.apply(lambda row: f'{row["SimpleQA_Coverage"]} ({row["SimpleQA_Coverage_SE"]})', axis=1)
df_perf_merged["SimpleQA_F1_disp"] = df_perf_merged.apply(lambda row: f'{row["SimpleQA_F1"]} ({row["SimpleQA_F1_SE"]})', axis=1)

In [127]:
df_perf_merged = df_perf_merged[["Judge Model","Prompt","GPQA_F1_disp", "GPQA_Cov_disp", "SimpleQA_F1_disp", "SimpleQA_Cov_disp"]].copy()

In [128]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1_disp,GPQA_Cov_disp,SimpleQA_F1_disp,SimpleQA_Cov_disp
0,claude-3-5-haiku-20241022,baseline,0.578 (0.027),0.778 (0.02),0.667 (0.033),0.458 (0.024)
1,claude-3-5-haiku-20241022,few,0.604 (0.034),0.438 (0.024),0.653 (0.033),0.45 (0.023)
2,claude-3-5-haiku-20241022,zero,0.648 (0.034),0.412 (0.023),0.673 (0.036),0.385 (0.022)
3,claude-3-5-sonnet-20241022,baseline,0.712 (0.023),0.748 (0.02),0.81 (0.025),0.502 (0.024)
4,claude-3-5-sonnet-20241022,few,0.716 (0.028),0.54 (0.023),0.654 (0.031),0.65 (0.022)
5,claude-3-5-sonnet-20241022,zero,0.738 (0.029),0.542 (0.024),0.733 (0.027),0.615 (0.022)
6,llama-4-maverick,baseline,0.774 (0.021),0.852 (0.016),0.673 (0.026),0.768 (0.02)
7,llama-4-maverick,few,0.751 (0.023),0.805 (0.018),0.692 (0.026),0.715 (0.021)
8,llama-4-maverick,zero,0.765 (0.025),0.618 (0.022),0.746 (0.029),0.528 (0.024)
9,llama-4-scout,baseline,0.702 (0.025),0.712 (0.021),0.58 (0.031),0.572 (0.023)


In [129]:
transformed_df_cost = df_cost.pivot_table(
        index=['Judge Model', 'Prompt'],  # These will be our row identifiers
        columns='Dataset',                # This will become our column headers
        values=['Mean Time', 'Mean Tokens Used'],       # These are the values we want to pivot
        aggfunc='first'                  # In case of duplicates, take the first value
    ).reset_index()

# Flatten the multi-level column headers
# This will create column names like ('Macro-F1', 'gqpa'), ('Macro-F1', 'simpleqa'), etc.
transformed_df_cost.columns = [f'{col[1]}_{col[0]}' if col[1] != '' else col[0] 
                            for col in transformed_df_cost.columns]
    
# Rename columns to match your desired format
column_mapping = {
    'Judge Model': 'Judge Model',
    'Prompt': 'Prompt',
    'gpqa_Mean Time': 'GPQA_Mean_Time',
    'gpqa_Mean Tokens Used': 'GPQA_Mean_Tokens',
    'simpleqa_Mean Time': 'SimpleQA_Mean_Time',
    'simpleqa_Mean Tokens Used': 'SimpleQA_Mean_Tokens',
}
    
transformed_df_cost = transformed_df_cost.rename(columns=column_mapping)
transformed_df_cost


,Judge Model,Prompt,GPQA_Mean_Time,SimpleQA_Mean_Time,GPQA_Mean_Tokens,SimpleQA_Mean_Tokens
0,claude-3-5-haiku-20241022,baseline,30.52 (4.09),15.81 (3.07),2641.07 (704.48),1014.98 (149.24)
1,claude-3-5-haiku-20241022,few,47.44 (4.69),40.01 (4.09),7673.44 (700.47),6435.11 (158.40)
2,claude-3-5-haiku-20241022,zero,43.12 (3.60),38.92 (3.14),4221.73 (685.85),3095.57 (131.72)
3,claude-3-5-sonnet-20241022,baseline,34.22 (6.08),19.22 (4.49),2914.80 (778.99),1190.43 (174.00)
4,claude-3-5-sonnet-20241022,few,52.92 (6.41),43.19 (5.45),8079.40 (745.71),6704.68 (161.86)
5,claude-3-5-sonnet-20241022,zero,53.91 (6.41),43.35 (6.44),4863.48 (731.27),3444.22 (171.12)
6,llama-4-maverick,baseline,65.91 (52.53),21.84 (13.56),6225.19 (1526.74),2008.57 (754.99)
7,llama-4-maverick,few,65.08 (41.43),41.29 (31.55),9945.70 (1444.02),6665.58 (512.60)
8,llama-4-maverick,zero,75.95 (44.90),36.17 (10.85),7492.54 (1357.27),4421.40 (448.00)
9,llama-4-scout,baseline,63.24 (28.92),17.02 (9.05),5403.00 (1833.85),1392.07 (434.73)


In [130]:
df_perf_merged = pd.merge(df_perf_merged, transformed_df_cost)

In [133]:
df_perf_merged

,Judge Model,Prompt,GPQA_F1_disp,GPQA_Cov_disp,SimpleQA_F1_disp,SimpleQA_Cov_disp,GPQA_Mean_Time,SimpleQA_Mean_Time,GPQA_Mean_Tokens,SimpleQA_Mean_Tokens
0,claude-3-5-haiku-20241022,baseline,0.578 (0.027),0.778 (0.02),0.667 (0.033),0.458 (0.024),30.52 (4.09),15.81 (3.07),2641.07 (704.48),1014.98 (149.24)
1,claude-3-5-haiku-20241022,few,0.604 (0.034),0.438 (0.024),0.653 (0.033),0.45 (0.023),47.44 (4.69),40.01 (4.09),7673.44 (700.47),6435.11 (158.40)
2,claude-3-5-haiku-20241022,zero,0.648 (0.034),0.412 (0.023),0.673 (0.036),0.385 (0.022),43.12 (3.60),38.92 (3.14),4221.73 (685.85),3095.57 (131.72)
3,claude-3-5-sonnet-20241022,baseline,0.712 (0.023),0.748 (0.02),0.81 (0.025),0.502 (0.024),34.22 (6.08),19.22 (4.49),2914.80 (778.99),1190.43 (174.00)
4,claude-3-5-sonnet-20241022,few,0.716 (0.028),0.54 (0.023),0.654 (0.031),0.65 (0.022),52.92 (6.41),43.19 (5.45),8079.40 (745.71),6704.68 (161.86)
5,claude-3-5-sonnet-20241022,zero,0.738 (0.029),0.542 (0.024),0.733 (0.027),0.615 (0.022),53.91 (6.41),43.35 (6.44),4863.48 (731.27),3444.22 (171.12)
6,llama-4-maverick,baseline,0.774 (0.021),0.852 (0.016),0.673 (0.026),0.768 (0.02),65.91 (52.53),21.84 (13.56),6225.19 (1526.74),2008.57 (754.99)
7,llama-4-maverick,few,0.751 (0.023),0.805 (0.018),0.692 (0.026),0.715 (0.021),65.08 (41.43),41.29 (31.55),9945.70 (1444.02),6665.58 (512.60)
8,llama-4-maverick,zero,0.765 (0.025),0.618 (0.022),0.746 (0.029),0.528 (0.024),75.95 (44.90),36.17 (10.85),7492.54 (1357.27),4421.40 (448.00)
9,llama-4-scout,baseline,0.702 (0.025),0.712 (0.021),0.58 (0.031),0.572 (0.023),63.24 (28.92),17.02 (9.05),5403.00 (1833.85),1392.07 (434.73)


In [137]:
final_df_perf = df_perf_merged[["Judge Model", "Prompt", "GPQA_F1_disp", "GPQA_Cov_disp", "GPQA_Mean_Time", "GPQA_Mean_Tokens", "SimpleQA_F1_disp", "SimpleQA_Cov_disp", "SimpleQA_Mean_Time", "SimpleQA_Mean_Tokens"]].copy()

In [140]:
perf_table_latex = final_df_perf.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:6.6g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Performance metrics for zeta using different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:finalperf",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(perf_table_latex, file=open("paper/perf_table.tex", "w+"))